In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F
@dp.table(
    name="intraday_bronze",
    comment="Bronze table for intraday data"
)
def intraday_bronze():
    df=spark.readStream.table('databricks_fundamentals.bronze.daily_stock_prices')
    return df


In [0]:
@dp.table(
    name="intraday_silver",
    comment="Silver table for intraday data"
)
def intraday_silver():
    bronze=spark.readStream.table('intraday_bronze')
    silver=bronze.withColumn('date',F.to_timestamp(F.col('date'))).withColumn('open',F.round(F.col('open').cast('double'),2)).withColumn('high',F.round(F.col('high').cast('double'),2)).withColumn('low',F.round(F.col('low').cast('double'),2)).withColumn('close',F.round(F.col('close').cast('double'),2)).withColumn('volume',F.col('volume').cast('long'))
    return silver

In [0]:
from pyspark.sql import Window
@dp.table(
    name="intraday_gold",
    comment="Gold table for intraday data"
)
def intraday_gold():
    silver=spark.read.table('intraday_silver')
    window=Window.partitionBy('symbol').orderBy('date')
    gold=silver.withColumn('price_change_5d', F.col('close') - F.lag(F.col('close'), 5).over(window)).withColumn('price_change_30d', F.col('close') - F.lag(F.col('close'), 30).over(window)).withColumn('price_change_90d', F.col('close') - F.lag(F.col('close'), 90).over(window)).withColumn('percentage_change_5d', F.round((F.col('price_change_5d') / F.lag(F.col('close'), 5).over(window)*100), 2)).withColumn('percentage_change_30d', F.round((F.col('price_change_30d') / F.lag(F.col('close'), 30).over(window))*100, 2)).withColumn('percentage_change_90d', F.round((F.col('price_change_90d') / F.lag(F.col('close'), 90).over(window))*100, 2)).withColumn('volume_change_5d', F.col('volume') - F.lag(F.col('volume'), 5).over(window)).withColumn('volume_change_30d', F.col('volume') - F.lag(F.col('volume'), 30).over(window)).withColumn('volume_change_90d', F.col('volume') - F.lag(F.col('volume'), 90).over(window))
    return gold